# T1 DeepAries — O1 변형 E: 10 epoch 학습 + 같은 환경 1 epoch 대조군 (시드 10 × 2 = 20회)

**사전 등록.** `notes/deeparies_e10_prereg.md`, 실행기 `replication/deeparies_run.py` @ `051eb94`(sha256 확인). 결과를 보기 전에 커밋한 노트북이다.

**공식 코드.** `dmis-lab/DeepAries` @ `634f559`. 수정 없이 실행하고, 인자는 `--seed`와 (e10만) `--train_epochs 10`이다.

**순서.** 시드 0–9마다 e1(1 epoch) → e10(10 epoch). 결과는 Drive `data/deeparies_e10/`에 저장된다.

**런타임: GPU(T4) 권장.** GPU가 없어도 돌지만 훨씬 느리다(CPU 기준 e1 약 20분, e10 최대 약 3.5시간). 끊기면 다시 연결하고 **모두 실행**하면 끝난 실행은 건너뛴다. 다 끝나면 Claude에게 "끝났어"라고 알려 주세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
OUT = pathlib.Path('/content/drive/MyDrive/data/deeparies_e10'); OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
COMMIT = '634f559cf4603eeb2ba7509e31319d076b21c16c'
SCRIPT_COMMIT, SCRIPT_SHA = '051eb94', '4c452b7fb451cefdbee0b2bdf45db345391562d918adf584b184664268aa44b0'
!mkdir -p /content/t1 /content/repl
!test -d /content/t1/DeepAries || git clone -q https://github.com/dmis-lab/DeepAries.git /content/t1/DeepAries
!cd /content/t1/DeepAries && git checkout -q $COMMIT && git log -1 --format='%H %cd'
!pip install -q yfinance timm einops
!wget -q -O /content/repl/deeparies_run.py https://raw.githubusercontent.com/winbean-kwon/ToBeRich/$SCRIPT_COMMIT/replication/deeparies_run.py
import hashlib; got = hashlib.sha256(open('/content/repl/deeparies_run.py', 'rb').read()).hexdigest()
assert got == SCRIPT_SHA, ('실행기 해시 불일치', got)
print('실행기 확인', SCRIPT_COMMIT, got[:12])

In [ ]:
import os, sys, json, subprocess, platform, time, torch, numpy
ENVV = dict(os.environ, DA_CODE='/content/t1/DeepAries', DA_RUNS=str(OUT), DA_THREADS=str(os.cpu_count()))
env_info = dict(python=platform.python_version(), torch=torch.__version__, numpy=numpy.__version__, cpus=os.cpu_count(),
                cuda=torch.cuda.is_available(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                time=time.strftime('%Y-%m-%d %H:%M'))
with open(OUT / 'env_info.jsonl', 'a') as f: f.write(json.dumps(env_info) + '\n')
print(env_info)
if not env_info['cuda']:
    print('주의: GPU 런타임이 아닙니다. 가능하면 런타임 유형을 T4 GPU로 바꾸세요(그대로 진행해도 됩니다).')

## 실행 20회 — 시드마다 e1 → e10 (끝난 실행은 건너뜀)

In [ ]:
(OUT / 'proc_logs').mkdir(exist_ok=True)
for s in range(10):
    for c in ('e1', 'e10'):
        if (OUT / f'W0_{c}_s{s}.json').exists():
            print(c, s, '완료됨'); continue
        t0 = time.time()
        p = subprocess.run([sys.executable, '/content/repl/deeparies_run.py', 'run', 'W0', c, str(s)],
                           capture_output=True, text=True, env=ENVV, cwd='/content/repl')
        (OUT / 'proc_logs' / f'{c}_s{s}.log').write_text(p.stdout + '\n--- stderr ---\n' + p.stderr)
        if p.returncode != 0:
            print(p.stdout[-2000:]); print(p.stderr[-3000:]); raise RuntimeError(f'{c} s{s} 실패')
        print(p.stdout.strip().splitlines()[-1], f'({(time.time() - t0) / 60:.1f}분)', time.strftime('%H:%M'), flush=True)
done = sum((OUT / f'W0_{c}_s{s}.json').exists() for s in range(10) for c in ('e1', 'e10'))
print(f'{done}/20 완료')